In [ ]:
!pip install darts -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score

# ====== CONFIG ======
PICKED_IPS_JSON = "/content/drive/MyDrive/Thesis/picked_ips.json"
TARGET_COL = "n_bytes"
MIN_POINTS = 50

TRAIN_RATIO = 0.35
VAL_RATIO   = 0.05


# ====== LOAD JSON ======
with open(PICKED_IPS_JSON, "r") as f:
    picked_files = json.load(f)

print(f"Loaded {len(picked_files)} processed IP CSV paths.")


# ====== MAIN LOOP ======
mean_results = []

for ip_path in picked_files:

    try:
        df = pd.read_csv(ip_path)
    except Exception as e:
        print(f"[SKIP] Could not read {ip_path}: {e}")
        continue

    # Use the processed 'time' column
    if "time" not in df.columns:
        print(f"[SKIP] No 'time' column in {ip_path}")
        continue

    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df = df.dropna(subset=["time"]).sort_values("time")

    # Fill missing values
    y = df[TARGET_COL].fillna(0).values

    N = len(y)
    if N < MIN_POINTS:
        print(f"[SKIP] Too few points in {ip_path}")
        continue

    # CESNET split
    train_end = int(TRAIN_RATIO * N)
    val_end   = int((TRAIN_RATIO + VAL_RATIO) * N)

    y_train = y[:train_end]
    y_test  = y[val_end:]

    if len(y_test) == 0:
        print(f"[SKIP] No test set for {ip_path}")
        continue

    # ------ MEAN MODEL ------
    mean_pred = np.mean(y_train)
    y_pred = np.full_like(y_test, mean_pred, dtype=float)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    mean_results.append({
        "ip_path": ip_path,
        "ip_name": Path(ip_path).stem,
        "rmse": rmse,
        "r2": r2,
        "train_size": len(y_train),
        "test_size": len(y_test),
        "mean_value": mean_pred
    })

    print(f"[OK] {Path(ip_path).name} — RMSE={rmse:.3f}  R2={r2:.3f}")

# Save results
mean_results_df = pd.DataFrame(mean_results)
mean_results_df.to_csv("/content/drive/MyDrive/Thesis/New_Results/Results_mean_processed_30IPs.csv", index=False)

mean_results_df


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
import os

out_dir = "/content/drive/MyDrive/Thesis/New_Results/Mean_evaluation"
os.makedirs(out_dir, exist_ok=True)

index = 6
ip_to_plot = mean_results_df.iloc[index]["ip_path"]

df = pd.read_csv(ip_to_plot)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time")

y = df["n_bytes"].fillna(0).values
N = len(y)

train_end = int(0.35 * N)
val_end   = int(0.40 * N)

y_train = y[:train_end]
y_test  = y[val_end:]

mean_pred = np.mean(y_train)

# build predictions for plotting
y_pred_full = np.concatenate([
    [np.nan] * val_end,            # no predictions on train/val block
    [mean_pred] * len(y_test)      # constant prediction for test
])

fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(df["time"], y, label="Actual (n_bytes)")
ax.plot(df["time"], y_pred_full, label="Mean Prediction", linestyle="--")

ax.set_title(f"Mean Model Forecast — {Path(ip_to_plot).name}")
ax.set_xlabel("Time")
ax.set_ylabel("Bytes")
ax.legend()
ax.grid(True)

fig.tight_layout()

out_path = os.path.join(out_dir, "21_mean_model_forecast.png")
fig.savefig(out_path, dpi=300, bbox_inches='tight')

plt.show()
plt.close(fig)

print("Saved to:", out_path)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from pathlib import Path

# --- define save folder ---
save_dir = "/content/drive/MyDrive/Thesis/New_Results/Mean_evaluation"
os.makedirs(save_dir, exist_ok=True)

# --- compute residuals ---
residuals = y[val_end:] - mean_pred
times_res = df["time"].iloc[val_end:]

# --- start figure ---
fig, ax = plt.subplots(figsize=(16, 4))

ax.plot(times_res, residuals, label="Residuals", linewidth=1.0)
ax.axhline(0, color="red", linestyle="--", linewidth=1.0, label="Zero Line")

ax.set_title(f"Residuals — {Path(ip_to_plot).name}", fontsize=14)
ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Residual (Actual - Mean)", fontsize=12)

ax.grid(True, which="both", linestyle="-", alpha=0.5)
ax.legend()
fig.tight_layout()

# --- save figure ---
out_name = f"{Path(ip_to_plot).stem}_mean_residuals.png"
out_path = os.path.join(save_dir, out_name)
fig.savefig(out_path, dpi=300, bbox_inches='tight')

plt.show()
plt.close(fig)

print("Saved residual plot to:", out_path)


In [ ]:
import matplotlib.pyplot as plt
import os
from pathlib import Path

# --- define save folder ---
save_dir = "/content/drive/MyDrive/Thesis/New_Results/Mean_evaluation"
os.makedirs(save_dir, exist_ok=True)

# --- histogram figure ---
fig, ax = plt.subplots(figsize=(6,4))

ax.hist(residuals, bins=50)
ax.set_title(f"Residual Histogram — {Path(ip_to_plot).name}", fontsize=12)
ax.set_xlabel("Residual")
ax.set_ylabel("Frequency")
ax.grid(True, linestyle="--", alpha=0.5)

fig.tight_layout()

# --- save the figure ---
out_name = f"{Path(ip_to_plot).stem}_mean_histogram.png"
out_path = os.path.join(save_dir, out_name)
fig.savefig(out_path, dpi=300, bbox_inches='tight')

plt.show()
plt.close(fig)

print("Saved histogram to:", out_path)
